# PCOS Quiz Model
**Goal:** estimate the likelihood of PCOS using only information a person can give at home (no blood tests, no ultrasound, no clinic measurements), so it could power a quiz.

**Plan in plain English**
1. Load and tidy the data
2. Look at it (class balance, which quiz features matter)
3. Compare three models with cross-validation
4. Choose a flagging threshold using the training data only
5. Test once on patients the model has never seen

> Put `PCOS_data.csv` next to this notebook (or change `DATA_PATH`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, recall_score, precision_score)

RANDOM_STATE = 42
DATA_PATH = "PCOS_data.csv"


## 1. Load and clean
The raw file has messy column names, two ID columns, and an empty extra column. We fix those.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

# Tidy column names: remove stray spaces
df.columns = (df.columns.str.strip()
                         .str.replace(r"\s+", " ", regex=True))

# Drop ID columns and the empty column
df = df.drop(columns=["Sl. No", "Patient File No."] + [c for c in df.columns if c.startswith("Unnamed")])

# Give the target a simple name
df = df.rename(columns={"PCOS (Y/N)": "PCOS"})

# Make sure every column is numeric (a few can be read as text in this dataset)
df = df.apply(pd.to_numeric, errors="coerce")

print("Clean shape:", df.shape)


## 2. Choose the quiz features
Only things a person can answer or measure at home: age, weight, height, waist/hip, cycle, symptoms and lifestyle. Everything that needs a lab, scan or clinic is left out.

Note: `Cycle(R/I)` is stored as numbers in this dataset (2 = regular, 4 = irregular, and a single 5).

In [ ]:
quiz_cols = ["Age (yrs)", "Weight (Kg)", "Height(Cm)", "BMI",
             "Cycle(R/I)", "Cycle length(days)",
             "Weight gain(Y/N)", "hair growth(Y/N)", "Skin darkening (Y/N)", "Hair loss(Y/N)", "Pimples(Y/N)",
             "Fast food (Y/N)", "Reg.Exercise(Y/N)",
             "Waist(inch)", "Hip(inch)", "Waist:Hip Ratio"]

y = df["PCOS"]
X = df[quiz_cols]

print("Missing values in quiz features:")
print(X.isna().sum()[X.isna().sum() > 0])


## 3. Explore
How many patients have PCOS, and which quiz features move with it?

In [ ]:
print(y.value_counts())
print((y.value_counts(normalize=True) * 100).round(1).astype(str) + " %")

corr = X.corrwith(y).sort_values(key=abs, ascending=False)

plt.figure(figsize=(7, 4.5))
corr.head(12)[::-1].plot(kind="barh", color="steelblue")
plt.title("Quiz features by correlation with PCOS")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()


**How to read this:** a bar to the right means the feature is higher in PCOS patients, to the left means lower.

## 4. Split the data
Keep 20% of patients aside as a final test. Stratify keeps the PCOS ratio the same in both parts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)


## 5. Compare models
We use **5-fold cross-validation** on the training data only. Each model gets the same preprocessing (fill gaps with the median, scale numbers).

Metrics (in plain English):
- **Recall:** of the people who truly have PCOS, how many did we catch? (most important for screening)
- **Precision:** of the people we flagged, how many really have it?
- **ROC-AUC:** how well the model ranks PCOS patients above non-PCOS patients (1.0 perfect, 0.5 coin flip)

In [ ]:
def make_models():
    prep = lambda: [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
    return {
        "Logistic Regression": Pipeline(prep() + [("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
        "KNN":                 Pipeline(prep() + [("clf", KNeighborsClassifier(n_neighbors=7))]),
        "Random Forest":       Pipeline(prep() + [("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                                               random_state=RANDOM_STATE))]),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = []
for name, model in make_models().items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"Model": name, **{m: s[f"test_{m}"].mean() for m in scoring}})

results = pd.DataFrame(rows).set_index("Model").round(3)
results


In [ ]:
# Pick the model with the best cross-validated ROC-AUC
best_name = results["roc_auc"].idxmax()
print("Best model:", best_name)


## 6. Choose the flagging threshold
By default a model flags someone as PCOS when its probability is 0.50 or higher. For screening, missing a real case is worse than a false alarm, so we lower the cutoff.

The threshold is chosen using **out-of-fold predictions on the training data only**, so the test set stays untouched. Change `target_recall` to catch more (lower threshold) or fewer (higher threshold) cases.

In [ ]:
model = make_models()[best_name]
cv_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

target_recall = 0.80
best_t = None
for t in np.arange(0.60, 0.10, -0.05):
    pred = (cv_proba >= t).astype(int)
    r = recall_score(y_train, pred)
    p = precision_score(y_train, pred)
    print(f"threshold {t:.2f} | recall {r:.2f} | precision {p:.2f}")
    if best_t is None and r >= target_recall:
        best_t = t

print("\nChosen threshold:", round(best_t, 2))


## 7. Final test on unseen patients
Fit on all training data, then test once on the held-out 20%, comparing the default 0.50 cutoff with the chosen one.

In [ ]:
model.fit(X_train, y_train)
test_proba = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 3), "\n")

for label, t in [("Default threshold", 0.5), ("Chosen threshold", best_t)]:
    pred = (test_proba >= t).astype(int)
    print(f"{label} {t:.2f} | recall {recall_score(y_test, pred):.2f} | "
          f"precision {precision_score(y_test, pred):.2f} | "
          f"false alarms {((pred == 1) & (y_test == 0)).sum()}")

final_pred = (test_proba >= best_t).astype(int)
print("\n", classification_report(y_test, final_pred, target_names=["No PCOS", "PCOS"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, final_pred),
                       display_labels=["No PCOS", "PCOS"]).plot(ax=ax[0], colorbar=False)
ax[0].set_title(f"Confusion matrix (threshold {best_t:.2f})")
RocCurveDisplay.from_predictions(y_test, test_proba, name=best_name, ax=ax[1])
ax[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
ax[1].set_title("ROC curve")
plt.tight_layout()
plt.show()


## 8. What drives the predictions?
Which quiz questions matter most (only available for Random Forest and Logistic Regression).

In [ ]:
clf = model.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    imp = pd.Series(clf.feature_importances_, index=X.columns)
elif hasattr(clf, "coef_"):
    imp = pd.Series(np.abs(clf.coef_[0]), index=X.columns)
else:
    imp = None
    print("Feature importance is not available for", best_name)

if imp is not None:
    imp = imp.sort_values(ascending=False).head(12)
    plt.figure(figsize=(7, 4.5))
    imp[::-1].plot(kind="barh", color="darkorange")
    plt.title(f"Top 12 quiz features ({best_name})")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


## 9. Likelihood for a patient
The model gives a probability, not just Yes/No. Here are the five highest-scoring patients in the test set, and whether they were flagged.

In [ ]:
out = pd.DataFrame({"Actual PCOS": y_test.values,
                    "Model score": test_proba.round(2),
                    "Flagged": test_proba >= best_t}, index=y_test.index)
out.sort_values("Model score", ascending=False).head(5)


## Notes and limitations
- Only ~540 patients from one source, and only 36 PCOS cases in the test set, so results can swing between splits. Treat numbers as indicative, not clinical.
- The model score is **not** a calibrated risk. Don't show it to users as "X% chance of PCOS". Use it only to flag "worth checking with a doctor".
- Lowering the threshold catches more cases but raises false alarms (roughly 1 in 3 flagged people may not have PCOS).
- This is a coursework model, **not** a medical diagnostic tool.
- Ideas to extend: hyperparameter tuning (GridSearchCV), repeated cross-validation for steadier estimates, SHAP for explanations.